# advanced domain adaption testing
- # Testing CDAN, MMD, CORAL, and Ensemble Methods


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import lightning as L
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_recall_curve
from sklearn.preprocessing import RobustScaler
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("ADVANCED DOMAIN ADAPTATION MODEL TESTING")
print("CDAN, MMD, CORAL, AND ENSEMBLE METHODS")
print("="*60)

# 2. data loading 

In [ ]:
# Load the same balanced datasets used in previous experiment
print("Loading balanced datasets...")
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')
ybt_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced_standardized.csv')

print(f"C4 balanced shape: {c4_balanced.shape}")
print(f"YBT balanced shape: {ybt_balanced.shape}")

# 3. feature engineering

In [ ]:
# Apply the same advanced feature engineering from previous experiment
def create_aggregate_features(df, prefix, n_items):
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items+1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    c4_balanced = create_aggregate_features(c4_balanced, prefix, n_items)
    ybt_balanced = create_aggregate_features(ybt_balanced, prefix, n_items)

# D-score
for df in [c4_balanced, ybt_balanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['d_score'] = df['eq_total'] - df['sqr_total']

# Age-EQ interaction
for df in [c4_balanced, ybt_balanced]:
    if 'age' in df.columns and 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']

# Age-AQ interaction
for df in [c4_balanced, ybt_balanced]:
    if 'age' in df.columns and 'aq_total' in df.columns:
        df['age_x_aq'] = df['age'] * df['aq_total']

# AQ-EQ interaction
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']

# EQ/SQR ratio
for df in [c4_balanced, ybt_balanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)

# Log-transformed AQ total
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns:
        df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))

# Square root of age
for df in [c4_balanced, ybt_balanced]:
    if 'age' in df.columns:
        df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))

# High AQ flag
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns:
        df['high_aq'] = (df['aq_total'] > 32).astype(int)

# Get common features
exclude_cols = ['autism_target']
c4_features = [col for col in c4_balanced.columns if col not in exclude_cols]
ybt_features = [col for col in ybt_balanced.columns if col not in exclude_cols]
common_features = sorted(list(set(c4_features) & set(ybt_features)))

print(f"Common features: {len(common_features)}")

# 4. data preparation

In [ ]:
# Prepare data arrays
X_c4 = c4_balanced[common_features].values
y_c4 = c4_balanced['autism_target'].values
X_ybt = ybt_balanced[common_features].values
y_ybt = ybt_balanced['autism_target'].values

scaler = RobustScaler()
X_c4_scaled = scaler.fit_transform(X_c4)
X_ybt_scaled = scaler.transform(X_ybt)

X_train, X_val, y_train, y_val = train_test_split(
    X_c4_scaled, y_c4, test_size=0.2, stratify=y_c4, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_ybt_scaled shape: {X_ybt_scaled.shape}")

# 5. CDAN

In [ ]:
print("\n" + "="*60)
print("CDAN (CONDITIONAL DOMAIN ADVERSARIAL NETWORK)")
print("="*60)

class CDANClassifier(L.LightningModule):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.3, 
                 learning_rate=0.0005, domain_weight=0.05):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Conditional domain discriminator (uses class predictions)
        self.domain_discriminator = nn.Sequential(
            nn.Linear(hidden_dims[-1] + 1, 128),  # +1 for class prediction
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self.learning_rate = learning_rate
        self.domain_weight = domain_weight
        self.class_criterion = nn.BCELoss()
        self.domain_criterion = nn.BCELoss()
        
    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        # Concatenate features with class predictions for conditional discriminator
        conditional_features = torch.cat([features, class_output], dim=1)
        domain_output = self.domain_discriminator(conditional_features)
        return class_output, domain_output
    
    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        total_loss = class_loss + self.domain_weight * domain_loss
        self.log('train_class_loss', class_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_domain_loss', domain_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_total_loss', total_loss, on_step=True, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        self.log('train_f1', f1, on_epoch=True, prog_bar=True)
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, domain_output = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        domain_loss = self.domain_criterion(domain_output.squeeze(), domain.float())
        total_loss = class_loss + self.domain_weight * domain_loss
        self.log('val_class_loss', class_loss, on_epoch=True, prog_bar=True)
        self.log('val_domain_loss', domain_loss, on_epoch=True, prog_bar=True)
        self.log('val_total_loss', total_loss, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        try:
            auc = roc_auc_score(y.cpu(), class_output.squeeze().detach().cpu())
        except ValueError:
            auc = 0.5
        self.log('val_f1', f1, on_epoch=True, prog_bar=True)
        self.log('val_auc', auc, on_epoch=True, prog_bar=True)
        return total_loss
    
    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.7, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_total_loss",
            },
        }

# 6. maximum mean discrepancy 

In [ ]:
print("\n" + "="*60)
print("MMD (MAXIMUM MEAN DISCREPANCY) DOMAIN ADAPTATION")
print("="*60)

def mmd_loss(source_features, target_features, kernel_mul=2.0, kernel_num=5):
    """Compute MMD loss between source and target features"""
    kernels = [kernel_mul ** i for i in range(kernel_num)]
    xx, yy, zz = 0, 0, 0
    
    for kernel in kernels:
        source_kernel = torch.exp(-torch.sum((source_features.unsqueeze(1) - source_features.unsqueeze(0)) ** 2, dim=2) / kernel)
        target_kernel = torch.exp(-torch.sum((target_features.unsqueeze(1) - target_features.unsqueeze(0)) ** 2, dim=2) / kernel)
        cross_kernel = torch.exp(-torch.sum((source_features.unsqueeze(1) - target_features.unsqueeze(0)) ** 2, dim=2) / kernel)
        
        xx += torch.mean(source_kernel)
        yy += torch.mean(target_kernel)
        zz += torch.mean(cross_kernel)
    
    return xx + yy - 2 * zz

class MMDClassifier(L.LightningModule):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.3, 
                 learning_rate=0.0005, mmd_weight=0.1):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self.learning_rate = learning_rate
        self.mmd_weight = mmd_weight
        self.class_criterion = nn.BCELoss()
        
    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        return class_output, features
    
    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, features = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Separate source and target features for MMD
        source_mask = (domain == 0)
        target_mask = (domain == 1)
        
        if torch.sum(source_mask) > 0 and torch.sum(target_mask) > 0:
            source_features = features[source_mask]
            target_features = features[target_mask]
            mmd_loss_val = mmd_loss(source_features, target_features)
            total_loss = class_loss + self.mmd_weight * mmd_loss_val
        else:
            total_loss = class_loss
            mmd_loss_val = torch.tensor(0.0)
        
        self.log('train_class_loss', class_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_mmd_loss', mmd_loss_val, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_total_loss', total_loss, on_step=True, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        self.log('train_f1', f1, on_epoch=True, prog_bar=True)
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, features = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # MMD for validation
        source_mask = (domain == 0)
        target_mask = (domain == 1)
        
        if torch.sum(source_mask) > 0 and torch.sum(target_mask) > 0:
            source_features = features[source_mask]
            target_features = features[target_mask]
            mmd_loss_val = mmd_loss(source_features, target_features)
            total_loss = class_loss + self.mmd_weight * mmd_loss_val
        else:
            total_loss = class_loss
            mmd_loss_val = torch.tensor(0.0)
        
        self.log('val_class_loss', class_loss, on_epoch=True, prog_bar=True)
        self.log('val_mmd_loss', mmd_loss_val, on_epoch=True, prog_bar=True)
        self.log('val_total_loss', total_loss, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        try:
            auc = roc_auc_score(y.cpu(), class_output.squeeze().detach().cpu())
        except ValueError:
            auc = 0.5
        self.log('val_f1', f1, on_epoch=True, prog_bar=True)
        self.log('val_auc', auc, on_epoch=True, prog_bar=True)
        return total_loss
    
    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.7, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_total_loss",
            },
        }

# 7. correlation alignment (CORAL)

In [ ]:
print("\n" + "="*60)
print("CORAL (CORRELATION ALIGNMENT) DOMAIN ADAPTATION")
print("="*60)

def coral_loss(source_features, target_features):
    """Compute CORAL loss between source and target features"""
    d = source_features.size(1)
    
    # Source covariance
    source_cov = torch.mm(source_features.t(), source_features) / (source_features.size(0) - 1)
    # Target covariance  
    target_cov = torch.mm(target_features.t(), target_features) / (target_features.size(0) - 1)
    
    # Frobenius norm of difference
    loss = torch.norm(source_cov - target_cov, p='fro') ** 2
    return loss / (4 * d * d)

class CORALClassifier(L.LightningModule):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout_rate=0.3, 
                 learning_rate=0.0005, coral_weight=0.1):
        super().__init__()
        self.save_hyperparameters()
        
        # Feature extractor
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        self.feature_extractor = nn.Sequential(*layers)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dims[-1], 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        self.learning_rate = learning_rate
        self.coral_weight = coral_weight
        self.class_criterion = nn.BCELoss()
        
    def forward(self, x):
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        return class_output, features
    
    def training_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, features = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # Separate source and target features for CORAL
        source_mask = (domain == 0)
        target_mask = (domain == 1)
        
        if torch.sum(source_mask) > 0 and torch.sum(target_mask) > 0:
            source_features = features[source_mask]
            target_features = features[target_mask]
            coral_loss_val = coral_loss(source_features, target_features)
            total_loss = class_loss + self.coral_weight * coral_loss_val
        else:
            total_loss = class_loss
            coral_loss_val = torch.tensor(0.0)
        
        self.log('train_class_loss', class_loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_coral_loss', coral_loss_val, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_total_loss', total_loss, on_step=True, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        self.log('train_f1', f1, on_epoch=True, prog_bar=True)
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        x, y, domain = batch
        class_output, features = self(x)
        class_loss = self.class_criterion(class_output.squeeze(), y.float())
        
        # CORAL for validation
        source_mask = (domain == 0)
        target_mask = (domain == 1)
        
        if torch.sum(source_mask) > 0 and torch.sum(target_mask) > 0:
            source_features = features[source_mask]
            target_features = features[target_mask]
            coral_loss_val = coral_loss(source_features, target_features)
            total_loss = class_loss + self.coral_weight * coral_loss_val
        else:
            total_loss = class_loss
            coral_loss_val = torch.tensor(0.0)
        
        self.log('val_class_loss', class_loss, on_epoch=True, prog_bar=True)
        self.log('val_coral_loss', coral_loss_val, on_epoch=True, prog_bar=True)
        self.log('val_total_loss', total_loss, on_epoch=True, prog_bar=True)
        y_pred = (class_output.squeeze() > 0.5).float()
        f1 = f1_score(y.cpu(), y_pred.cpu(), average='weighted', zero_division=0)
        try:
            auc = roc_auc_score(y.cpu(), class_output.squeeze().detach().cpu())
        except ValueError:
            auc = 0.5
        self.log('val_f1', f1, on_epoch=True, prog_bar=True)
        self.log('val_auc', auc, on_epoch=True, prog_bar=True)
        return total_loss
    
    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.7, patience=3
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_total_loss",
            },
        }

# 8. data module

In [ ]:
print("\n" + "="*60)
print("DOMAIN-AWARE DATA MODULE")
print("="*60)

class DomainAdaptationDataModule(L.LightningDataModule):
    def __init__(self, X_train, X_val, y_train, y_val, X_target, y_target, batch_size=64):
        super().__init__()
        self.X_train = X_train
        self.X_val = X_val
        self.y_train = y_train
        self.y_val = y_val
        self.X_target = X_target
        self.y_target = y_target
        self.batch_size = batch_size
    
    def setup(self, stage=None):
        self.train_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_train),
            torch.LongTensor(self.y_train),
            torch.zeros(len(self.X_train))
        )
        self.val_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_val),
            torch.LongTensor(self.y_val),
            torch.zeros(len(self.X_val))
        )
        self.target_dataset = torch.utils.data.TensorDataset(
            torch.FloatTensor(self.X_target),
            torch.LongTensor(self.y_target),
            torch.ones(len(self.X_target))
        )
    
    def train_dataloader(self):
        combined_dataset = torch.utils.data.ConcatDataset([
            self.train_dataset, self.target_dataset
        ])
        return torch.utils.data.DataLoader(
            combined_dataset, 
            batch_size=self.batch_size, 
            shuffle=True,
            num_workers=4,
            pin_memory=True
        )
    
    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            shuffle=False,
            num_workers=4,
            pin_memory=True
        )

# 9. model training function 

In [ ]:
def train_and_evaluate_model(model_class, model_name, data_module, max_epochs=50):
    """Train and evaluate a domain adaptation model"""
    print(f"\nTraining {model_name}...")
    
    input_dim = len(common_features)
    model = model_class(input_dim=input_dim)
    
    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator='auto',
        devices=1,
        callbacks=[
            L.pytorch.callbacks.EarlyStopping(
                monitor='val_f1',
                patience=10,
                mode='max',
                verbose=True
            ),
            L.pytorch.callbacks.ModelCheckpoint(
                monitor='val_f1',
                mode='max',
                save_top_k=1,
                filename=f'best_{model_name.lower()}_{{epoch:02d}}_{{val_f1:.3f}}',
                verbose=True
            ),
            L.pytorch.callbacks.LearningRateMonitor(logging_interval='epoch')
        ],
        log_every_n_steps=25,
        enable_progress_bar=True,
        enable_model_summary=False,
        deterministic=True
    )
    
    trainer.fit(model, data_module)
    
    # Load best model and evaluate
    best_model_path = trainer.checkpoint_callback.best_model_path
    model = model_class.load_from_checkpoint(best_model_path)
    model.eval()
    
    # Validation performance
    val_predictions = []
    val_probs = []
    val_targets = []
    
    with torch.no_grad():
        for batch in data_module.val_dataloader():
            x, y, domain = batch
            device = next(model.parameters()).device
            x = x.to(device)
            if model_name in ['CDAN', 'MMD', 'CORAL']:
                class_output, _ = model(x)
            else:
                class_output = model(x)
            val_probs.extend(class_output.squeeze().cpu().numpy())
            val_predictions.extend((class_output.squeeze() > 0.5).cpu().numpy())
            val_targets.extend(y.cpu().numpy())
    
    val_probs = np.array(val_probs)
    val_predictions = np.array(val_predictions)
    val_targets = np.array(val_targets)
    
    val_f1 = f1_score(val_targets, val_predictions, average='weighted')
    val_auc = roc_auc_score(val_targets, val_probs)
    
    # YBT performance
    X_ybt_tensor = torch.FloatTensor(X_ybt_scaled)
    ybt_dataset = torch.utils.data.TensorDataset(
        X_ybt_tensor, torch.LongTensor(y_ybt), torch.ones(len(X_ybt_scaled))
    )
    ybt_dataloader = torch.utils.data.DataLoader(ybt_dataset, batch_size=64, shuffle=False)
    
    ybt_predictions = []
    ybt_probs = []
    ybt_targets = []
    
    device = next(model.parameters()).device
    with torch.no_grad():
        for batch in ybt_dataloader:
            x, y, domain = batch
            x = x.to(device)
            if model_name in ['CDAN', 'MMD', 'CORAL']:
                class_output, _ = model(x)
            else:
                class_output = model(x)
            ybt_probs.extend(class_output.squeeze().cpu().numpy())
            ybt_predictions.extend((class_output.squeeze() > 0.5).cpu().numpy())
            ybt_targets.extend(y.cpu().numpy())
    
    ybt_probs = np.array(ybt_probs)
    ybt_predictions = np.array(ybt_predictions)
    ybt_targets = np.array(ybt_targets)
    
    # Threshold optimization
    prec, rec, thresholds = precision_recall_curve(ybt_targets, ybt_probs)
    f1s = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_thresh_idx = np.argmax(f1s)
    best_threshold = thresholds[best_thresh_idx]
    
    ybt_predictions_optimal = (ybt_probs >= best_threshold).astype(int)
    ybt_f1 = f1_score(ybt_targets, ybt_predictions_optimal, average='weighted')
    ybt_auc = roc_auc_score(ybt_targets, ybt_probs)
    
    results = {
        'model': model_name,
        'val_f1': val_f1,
        'val_auc': val_auc,
        'ybt_f1': ybt_f1,
        'ybt_auc': ybt_auc,
        'best_threshold': best_threshold,
        'epochs': trainer.current_epoch
    }
    
    print(f"{model_name} Results:")
    print(f"  Validation F1: {val_f1:.3f}")
    print(f"  Validation AUC: {val_auc:.3f}")
    print(f"  YBT F1: {ybt_f1:.3f}")
    print(f"  YBT AUC: {ybt_auc:.3f}")
    print(f"  Best threshold: {best_threshold:.3f}")
    print(f"  Epochs trained: {trainer.current_epoch}")
    
    return results, model

# 10. train all models

In [ ]:
print("\n" + "="*60)
print("TRAINING ALL DOMAIN ADAPTATION MODELS")
print("="*60)

# Create data module
data_module = DomainAdaptationDataModule(
    X_train, X_val, y_train, y_val, X_ybt_scaled, y_ybt, batch_size=64
)

# Train all models
models_to_test = [
    (CDANClassifier, 'CDAN'),
    (MMDClassifier, 'MMD'),
    (CORALClassifier, 'CORAL')
]

all_results = []

for model_class, model_name in models_to_test:
    try:
        results, model = train_and_evaluate_model(model_class, model_name, data_module)
        all_results.append(results)
        
        # Save model
        os.makedirs('/Users/eb2007/playground/bullpy/c4_play2/models', exist_ok=True)
        torch.save(model.state_dict(), f'/Users/eb2007/playground/bullpy/c4_play2/models/{model_name.lower()}.pth')
        
    except Exception as e:
        print(f"Error training {model_name}: {e}")
        continue

print("\nAll models trained successfully!")

# 11. ensemble methods

In [ ]:
print("\n" + "="*60)
print("ENSEMBLE METHODS")
print("="*60)

# Load all trained models
models = {}
for model_class, model_name in models_to_test:
    try:
        model_path = f'/Users/eb2007/playground/bullpy/c4_play2/models/{model_name.lower()}.pth'
        if os.path.exists(model_path):
            model = model_class(input_dim=len(common_features))
            model.load_state_dict(torch.load(model_path))
            model.eval()
            models[model_name] = model
    except Exception as e:
        print(f"Error loading {model_name}: {e}")

# Ensemble predictions
def get_ensemble_predictions(models, dataloader, ensemble_method='average'):
    all_probs = []
    
    for model_name, model in models.items():
        model_probs = []
        device = next(model.parameters()).device
        
        with torch.no_grad():
            for batch in dataloader:
                x, y, domain = batch
                x = x.to(device)
                if model_name in ['CDAN', 'MMD', 'CORAL']:
                    class_output, _ = model(x)
                else:
                    class_output = model(x)
                model_probs.extend(class_output.squeeze().cpu().numpy())
        
        all_probs.append(np.array(model_probs))
    
    # Ensemble methods
    if ensemble_method == 'average':
        ensemble_probs = np.mean(all_probs, axis=0)
    elif ensemble_method == 'weighted':
        # Weight by validation performance
        weights = [results['val_f1'] for results in all_results if results['model'] in models.keys()]
        weights = np.array(weights) / np.sum(weights)
        ensemble_probs = np.average(all_probs, axis=0, weights=weights)
    elif ensemble_method == 'max':
        ensemble_probs = np.max(all_probs, axis=0)
    elif ensemble_method == 'min':
        ensemble_probs = np.min(all_probs, axis=0)
    
    return ensemble_probs

# Test ensemble on YBT
X_ybt_tensor = torch.FloatTensor(X_ybt_scaled)
ybt_dataset = torch.utils.data.TensorDataset(
    X_ybt_tensor, torch.LongTensor(y_ybt), torch.ones(len(X_ybt_scaled))
)
ybt_dataloader = torch.utils.data.DataLoader(ybt_dataset, batch_size=64, shuffle=False)

ensemble_methods = ['average', 'weighted', 'max', 'min']
ensemble_results = []

for method in ensemble_methods:
    try:
        ensemble_probs = get_ensemble_predictions(models, ybt_dataloader, method)
        
        # Threshold optimization
        prec, rec, thresholds = precision_recall_curve(y_ybt, ensemble_probs)
        f1s = 2 * (prec * rec) / (prec + rec + 1e-8)
        best_thresh_idx = np.argmax(f1s)
        best_threshold = thresholds[best_thresh_idx]
        
        ensemble_predictions = (ensemble_probs >= best_threshold).astype(int)
        ensemble_f1 = f1_score(y_ybt, ensemble_predictions, average='weighted')
        ensemble_auc = roc_auc_score(y_ybt, ensemble_probs)
        
        ensemble_results.append({
            'method': f'Ensemble_{method}',
            'f1': ensemble_f1,
            'auc': ensemble_auc,
            'threshold': best_threshold
        })
        
        print(f"Ensemble {method}: F1={ensemble_f1:.3f}, AUC={ensemble_auc:.3f}")
        
    except Exception as e:
        print(f"Error with ensemble {method}: {e}")

all_results.extend(ensemble_results)

# 12. results comparison

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE RESULTS COMPARISON")
print("="*60)

# Previous DANN results for comparison
previous_results = {
    'DANN_Improved': {'f1': 0.729, 'auc': 0.787, 'threshold': 0.421}
}

# Create comparison dataframe
comparison_data = []
for result in all_results:
    if 'Ensemble' in result.get('method', result.get('model', '')):
        comparison_data.append({
            'Model': result['method'],
            'F1_Score': result['f1'],
            'ROC_AUC': result['auc'],
            'Threshold': result['threshold']
        })
    else:
        comparison_data.append({
            'Model': result['model'],
            'F1_Score': result['ybt_f1'],
            'ROC_AUC': result['ybt_auc'],
            'Threshold': result['best_threshold']
        })

# Add previous DANN results
comparison_data.append({
    'Model': 'DANN_Improved',
    'F1_Score': previous_results['DANN_Improved']['f1'],
    'ROC_AUC': previous_results['DANN_Improved']['auc'],
    'Threshold': previous_results['DANN_Improved']['threshold']
})

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('F1_Score', ascending=False)

print("\nPerformance Comparison (sorted by F1 Score):")
print(comparison_df)

# Find best model
best_model = comparison_df.iloc[0]
print(f"\nBest performing model: {best_model['Model']}")
print(f"F1 Score: {best_model['F1_Score']:.3f}")
print(f"ROC-AUC: {best_model['ROC_AUC']:.3f}")

# Plot results
plt.figure(figsize=(12, 8))
plt.subplot(2, 2, 1)
plt.bar(comparison_df['Model'], comparison_df['F1_Score'])
plt.title('F1 Scores Comparison')
plt.xticks(rotation=45)
plt.ylabel('F1 Score')

plt.subplot(2, 2, 2)
plt.bar(comparison_df['Model'], comparison_df['ROC_AUC'])
plt.title('ROC-AUC Comparison')
plt.xticks(rotation=45)
plt.ylabel('ROC-AUC')

plt.subplot(2, 2, 3)
plt.scatter(comparison_df['F1_Score'], comparison_df['ROC_AUC'])
for i, row in comparison_df.iterrows():
    plt.annotate(row['Model'], (row['F1_Score'], row['ROC_AUC']), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)
plt.xlabel('F1 Score')
plt.ylabel('ROC-AUC')
plt.title('F1 vs AUC')

plt.subplot(2, 2, 4)
plt.bar(comparison_df['Model'], comparison_df['Threshold'])
plt.title('Optimal Thresholds')
plt.xticks(rotation=45)
plt.ylabel('Threshold')

plt.tight_layout()
plt.show()

print(f"\nFinal Summary:")
print(f"Best model: {best_model['Model']}")
print(f"Best F1: {best_model['F1_Score']:.3f}")
print(f"Best AUC: {best_model['ROC_AUC']:.3f}")
print(f"Improvement over DANN: {best_model['F1_Score'] - previous_results['DANN_Improved']['f1']:.3f}")